In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np
ROOT=Path('/content/drive/MyDrive/US_ETF'); RES=ROOT/'model_lab_v1/results/open_revalidation_v1'; CACHE=ROOT/'directional_research/open_revalidation_1m_alpaca_v1/iex'; AUDIT=RES/'open_revalidation_trade_audit.parquet'
df=pd.read_parquet(AUDIT); df['entry_timestamp']=pd.to_datetime(df['entry_timestamp'],utc=True,errors='coerce'); df['exit_timestamp']=pd.to_datetime(df['exit_timestamp'],utc=True,errors='coerce')
price=['entry_price_iex','fixed4_exit_price_iex','prev_close_price_iex','open_0_price_iex','open_5_price_iex','open_15_price_iex']
derived=['position_return_prev_close','position_return_open','position_return_5m','position_return_15m','overnight_gap_return','open_momentum_5m','open_momentum_15m','giveback_prev_close_to_5m','reconstructed_fixed4_raw_return','cost_proxy','reconstructed_fixed4_net_return']
pol=[c for c in df.columns if c.startswith('OPEN_')]
print('='*100); print('KALMAN FINAL GAP + DERIVED AUDIT v1.9 — READ ONLY'); print('='*100)
print('rows=',len(df),'ready=',int(df[price].notna().all(axis=1).sum()))
print('\n[PRICE MISSING]'); print(df[price].isna().sum().to_string())
print('\n[DERIVED MISSING]'); print(df[derived].isna().sum().to_string())
print('\n[OPEN POLICY MISSING]'); print(df[pol].isna().sum().to_string())

miss=df[df['fixed4_exit_price_iex'].isna()].copy()
print('\n[FINAL FIXED4 GAP ROW]')
show=['policy','fold','symbol','entry_timestamp','exit_timestamp','entry_effective_ts','fixed4_effective_ts','entry_seq','exit_seq','holding_bars','exit_reason','fixed4_exit_price_iex','backfill_error']
print(miss[[c for c in show if c in miss.columns]].to_string(index=False))

print('\n[BAR COVERAGE AROUND FINAL GAP]')
for _,r in miss.iterrows():
    s=str(r.symbol).upper(); fs=sorted((CACHE/s).glob('*.parquet')); chunks=[]
    for f in fs:
        try:
            x=pd.read_parquet(f); tc=next((c for c in ['timestamp','time','datetime','t'] if c in x.columns),None)
            if tc:
                x=x.copy(); x['timestamp']=pd.to_datetime(x[tc],utc=True,errors='coerce'); chunks.append(x)
        except: pass
    if chunks:
        z=pd.concat(chunks,ignore_index=True).dropna(subset=['timestamp']).drop_duplicates('timestamp').sort_values('timestamp')
        t=r.exit_timestamp; near=z[(z.timestamp>=t-pd.Timedelta(minutes=30))&(z.timestamp<=t+pd.Timedelta(minutes=30))]
        print('symbol=',s,'exit=',t,'bars_near=',len(near),'cache_files=',len(fs))
        print(near.tail(20).to_string(index=False))
    else: print('symbol=',s,'NO CACHE')

derived_complete=df[derived].notna().all(axis=1)
policy_complete=df[pol].notna().all(axis=1) if pol else pd.Series(False,index=df.index)
print('\n[ANALYSIS READINESS]')
print('price_complete=',int(df[price].notna().all(axis=1).sum()),'/',len(df))
print('derived_complete=',int(derived_complete.sum()),'/',len(df))
print('policy_complete=',int(policy_complete.sum()),'/',len(df))
print('\nNo files modified. Do NOT run fold/bootstrap on stale derived/policy columns if completeness is below price completeness.')
